In [1]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

import os
import json
import re
from transformers import AutoTokenizer

[nltk_data] Downloading package punkt to /u/adityav/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /u/adityav/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
def get_all_data():
    with open('/var/local/adityav/Projects/temp/mlwb/final_project/transcript_books.json', 'r') as f:
        all_data = json.load(f)
    return all_data

def sentence_based_chunking(text, max_sentences):
    if type(text) == list:
        text = '\n'.join(text)
    sentences = nltk.sent_tokenize(text)
    chunks = []
    current_chunk = []
    
    for sentence in sentences:
        if len(current_chunk) < max_sentences:
            current_chunk.append(sentence)
        else:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentence]
    
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    
    return chunks


def token_based_chunking(text, max_tokens):
    if type(text) == list:
        text = '\n'.join(text)
        
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    tokens = tokenizer.tokenize(text)
    chunks = []
    current_chunk = []
    
    for token in tokens:
        if len(current_chunk) < max_tokens:
            current_chunk.append(token)
        else:
            chunks.append(tokenizer.convert_tokens_to_string(current_chunk))
            current_chunk = [token]
    
    if current_chunk:
        chunks.append(tokenizer.convert_tokens_to_string(current_chunk))
    
    return chunks

In [3]:
def save_chunks(all_data, save_file):
    all_chunks = []
    ctr = 1
    for data_type, data in all_data.items():
        for d in data:
            all_chunks.append({'id': f"chunk_{ctr}", 'chunk': d})
            ctr += 1
    with open(save_file, 'w') as f:
        json.dump(all_chunks, f)

In [4]:
all_data = get_all_data()
print(all_data.keys())

dict_keys(['merged_transcript.txt', 'Transcript_Feb19_Mar5.txt', 'Bishop-Pattern-Recognition-and-Machine-Learning-2006.txt', 'understanding-machine-learning-theory-algorithms.txt', 'ESLII_print12_toc.txt', 'ProbabilisticMachineLearningAnIntroduction.txt', 'ProbabilisticMachineLearningAdvancedTopics.txt', 'ISLP_website.txt'])


In [65]:
outfile = "chunked_transcript_books_10sentences.json"
chunked_data = {}
total = 0
for data_type, data in all_data.items():
    print(f"Chunking file {data_type}")
    chunks = sentence_based_chunking(all_data[data_type], 10)
    print(f"Chunks: {len(chunks)}")
    chunked_data[data_type] = chunks
    total += len(chunks)
print(f"Total chunks: {total}")
# with open(outfile, 'w') as f:
#     json.dump(chunked_data, f)
save_chunks(chunked_data, outfile)

Chunking file merged_transcript.txt
Chunks: 690
Chunking file Transcript_Feb19_Mar5.txt
Chunks: 321
Chunking file Bishop-Pattern-Recognition-and-Machine-Learning-2006.txt
Chunks: 1179
Chunking file understanding-machine-learning-theory-algorithms.txt
Chunks: 887
Chunking file ESLII_print12_toc.txt
Chunks: 1274
Chunking file ProbabilisticMachineLearningAnIntroduction.txt
Chunks: 1484
Chunking file ProbabilisticMachineLearningAdvancedTopics.txt
Chunks: 2410
Chunking file ISLP_website.txt
Chunks: 1212
Total chunks: 9457


In [5]:
outfile = "chunked_transcript_books_512berttokens.json"
chunked_data = {}
total = 0
for data_type, data in all_data.items():
    print(f"Chunking file {data_type}")
    chunks = token_based_chunking(all_data[data_type], 512)
    print(f"Chunks: {len(chunks)}")
    chunked_data[data_type] = chunks
    total += len(chunks)
print(f"Total chunks: {total}")
# with open(outfile, 'w') as f:
#     json.dump(chunked_data, f)

save_chunks(chunked_data, outfile)

Chunking file merged_transcript.txt


Token indices sequence length is longer than the specified maximum sequence length for this model (83802 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (37859 > 512). Running this sequence through the model will result in indexing errors


Chunks: 164
Chunking file Transcript_Feb19_Mar5.txt
Chunks: 74
Chunking file Bishop-Pattern-Recognition-and-Machine-Learning-2006.txt


Token indices sequence length is longer than the specified maximum sequence length for this model (410264 > 512). Running this sequence through the model will result in indexing errors


Chunks: 802
Chunking file understanding-machine-learning-theory-algorithms.txt


...

Chunks: 921
Chunking file ProbabilisticMachineLearningAdvancedTopics.txt


Token indices sequence length is longer than the specified maximum sequence length for this model (825011 > 512). Running this sequence through the model will result in indexing errors


Chunks: 1612
Chunking file ISLP_website.txt


Token indices sequence length is longer than the specified maximum sequence length for this model (356424 > 512). Running this sequence through the model will result in indexing errors


Chunks: 697
Total chunks: 5775


In [6]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

embed_model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
semantic_chunker = SemanticChunker(embed_model, breakpoint_threshold_type="percentile")

/tmp/ipykernel_3165543/1163242439.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embed_model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
2025-05-02 15:19:23.124558: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-05-02 15:19:23.124604: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


[2025-05-02 15:19:24,478] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile
collect2: error: ld returned 1 exit status


In [14]:
semantic_chunks = semantic_chunker.create_documents([' '.join(all_data['merged_transcript.txt'])])
print(len(semantic_chunks))

377


In [ ]:
def semantic_chunk(data):
    embed_model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
    semantic_chunker = SemanticChunker(embed_model, breakpoint_threshold_type="percentile")
    semantic_chunks = semantic_chunker.create_documents([' '.join(data)])
    semantic_chunks = [i.page_content for i in semantic_chunks]
    return semantic_chunks

In [21]:
outfile = "chunked_transcript_books_semantic.json"
chunked_data = {}
total = 0
for data_type, data in all_data.items():
    print(f"Chunking file {data_type}")
    chunks = semantic_chunk(all_data[data_type])
    print(f"Chunks: {len(chunks)}")
    chunked_data[data_type] = chunks
    total += len(chunks)
print(f"Total chunks: {total}")

save_chunks(chunked_data, outfile)

Chunking file merged_transcript.txt
Chunks: 377
Chunking file Transcript_Feb19_Mar5.txt
Chunks: 176
Chunking file Bishop-Pattern-Recognition-and-Machine-Learning-2006.txt
Chunks: 609
Chunking file understanding-machine-learning-theory-algorithms.txt
Chunks: 466
Chunking file ESLII_print12_toc.txt
Chunks: 3450
Chunking file ProbabilisticMachineLearningAnIntroduction.txt
Chunks: 760
Chunking file ProbabilisticMachineLearningAdvancedTopics.txt
Chunks: 1228
Chunking file ISLP_website.txt
Chunks: 615
Total chunks: 7681
